In [37]:
import pandas as pd

In [38]:
df = pd.read_csv('data/train.csv')
df.shape
df.head(1)

(10000, 17)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,extra,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,0,2023-06-28 17:31:46,2023-06-28 18:22:12,1.0,1.5,1.0,N,212,237,Credit Card,5.0,6.53321,0.0,1.0,24.8,2.5,0.0


In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   VendorID               10000 non-null  int64  
 1   tpep_pickup_datetime   10000 non-null  object 
 2   tpep_dropoff_datetime  10000 non-null  object 
 3   passenger_count        9634 non-null   float64
 4   trip_distance          10000 non-null  float64
 5   RatecodeID             9634 non-null   float64
 6   store_and_fwd_flag     9634 non-null   object 
 7   PULocationID           10000 non-null  int64  
 8   DOLocationID           10000 non-null  int64  
 9   payment_type           10000 non-null  object 
 10  extra                  10000 non-null  float64
 11  tip_amount             10000 non-null  float64
 12  tolls_amount           10000 non-null  float64
 13  improvement_surcharge  10000 non-null  float64
 14  total_amount           10000 non-null  float64
 15  con

In [40]:
num_cols = df.select_dtypes(include='number').columns.to_list()
cat_cols = df.select_dtypes(exclude='number').columns.to_list()

df['store_and_fwd_flag'].value_counts()
df['payment_type'].value_counts()

store_and_fwd_flag
N    9565
Y      69
Name: count, dtype: int64

payment_type
Credit Card    7727
Cash           1706
Wallet          366
unknown         133
UPI              68
Name: count, dtype: int64

In [41]:
df.describe()

,VendorID,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,extra,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
count,10000.000000,9634.000000,10000.000000,9634.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,9634.000000,9634.000000
mean,0.729100,1.357276,3.679527,1.450695,132.710800,132.33240,1.940950,6.094658,0.664425,0.979490,29.740157,2.238426,0.161122
std,0.444672,0.883676,4.905798,5.988978,75.789597,75.95944,1.945885,4.438894,2.441070,0.200076,26.256398,0.834189,0.512859
min,0.000000,0.000000,0.000000,1.000000,1.000000,1.00000,-7.500000,0.000713,-26.550000,-1.000000,-129.300000,-2.500000,-1.750000
25%,0.000000,1.000000,1.070000,1.000000,67.000000,68.00000,0.000000,3.466789,0.000000,1.000000,16.300000,2.500000,0.000000
50%,1.000000,1.000000,1.820000,1.000000,133.000000,132.00000,1.750000,5.208233,0.000000,1.000000,21.360000,2.500000,0.000000
75%,1.000000,1.000000,3.630000,1.000000,198.000000,199.00000,2.500000,7.455228,0.000000,1.000000,31.800000,2.500000,0.000000
max,2.000000,6.000000,71.940000,99.000000,264.000000,264.00000,11.750000,84.032617,32.050000,1.000000,551.000000,2.500000,1.750000


In [42]:
df.isna().sum()

VendorID                   0
tpep_pickup_datetime       0
tpep_dropoff_datetime      0
passenger_count          366
trip_distance              0
RatecodeID               366
store_and_fwd_flag       366
PULocationID               0
DOLocationID               0
payment_type               0
extra                      0
tip_amount                 0
tolls_amount               0
improvement_surcharge      0
total_amount               0
congestion_surcharge     366
Airport_fee              366
dtype: int64

In [43]:
df['tpep_pickup_datetime'].dtype
df['tpep_dropoff_datetime'].dtype

df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

df['tpep_pickup_datetime'].head(1)
df['tpep_dropoff_datetime'].head(1)

dtype('O')

dtype('O')

0   2023-06-28 17:31:46
Name: tpep_pickup_datetime, dtype: datetime64[ns]

0   2023-06-28 18:22:12
Name: tpep_dropoff_datetime, dtype: datetime64[ns]

In [44]:
df['passenger_count'].value_counts()
df['RatecodeID'].value_counts()

passenger_count
1.0    7282
2.0    1424
3.0     357
4.0     197
0.0     173
5.0     119
6.0      82
Name: count, dtype: int64

RatecodeID
1.0     9057
2.0      401
3.0       58
5.0       51
99.0      36
4.0       31
Name: count, dtype: int64

In [45]:
for col in num_cols:
  col_mean = df[col].mean()
  df[col] = df[col].fillna(col_mean)

df['store_and_fwd_flag'] = df['store_and_fwd_flag'].fillna(df['store_and_fwd_flag'].mode()[0])

df.isna().sum().sum()

np.int64(0)

In [46]:
for col in num_cols:
  q1 = df[col].quantile(0.25)
  q3 = df[col].quantile(0.75)

  IQR = q3 - q1

  lower_bound = df[col] - 1.5 * IQR
  upper_bound = df[col] + 1.5 * IQR

  print((df.loc[(df[col] < lower_bound) | (
      df[col] > upper_bound), col]))

Series([], Name: VendorID, dtype: int64)
Series([], Name: passenger_count, dtype: float64)
Series([], Name: trip_distance, dtype: float64)
Series([], Name: RatecodeID, dtype: float64)
Series([], Name: PULocationID, dtype: int64)
Series([], Name: DOLocationID, dtype: int64)
Series([], Name: extra, dtype: float64)
Series([], Name: tip_amount, dtype: float64)
Series([], Name: tolls_amount, dtype: float64)
Series([], Name: improvement_surcharge, dtype: float64)
Series([], Name: total_amount, dtype: float64)
Series([], Name: congestion_surcharge, dtype: float64)
Series([], Name: Airport_fee, dtype: float64)


In [47]:
from sklearn.model_selection import train_test_split

X = df.drop('total_amount', axis=1)
y = df['total_amount']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train.shape
X_test.shape
y_train.shape
y_test.shape

(8000, 16)

(2000, 16)

(8000,)

(2000,)

In [57]:
from sklearn.preprocessing import StandardScaler

scalar_cols = ['VendorID',
               'passenger_count',
               'trip_distance',
               'RatecodeID',
               'PULocationID',
               'DOLocationID',
               'extra',
               'tip_amount',
               'tolls_amount',
               'improvement_surcharge',
               'congestion_surcharge',
               'Airport_fee']

scalar = StandardScaler()
X_train[scalar_cols] = scalar.fit_transform(X_train[scalar_cols])
X_test[scalar_cols] = scalar.transform(X_test[scalar_cols])

In [50]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV

In [59]:
# CV = GridSearchCV(
#     Ridge(),
#     {'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]},
#     cv=5, scoring='r2', n_jobs=-1, error_score=
# )

# CV.fit(X_train, y_train)
# y_pred = CV.best_params_.predict(X_test)
# r2_score(y_test, y_pred)